# Notebook 01 — Attention Intuition

## Learning Objectives
- Understand *why* attention is needed: words have context-dependent meaning.
- Manually compute attention scores using the query / key / value metaphor.
- Visualize which words a given word attends to using a heatmap.
- Compute cosine similarity between word vectors to measure semantic closeness.
- Connect the intuition to the scaled dot-product attention formula.

## Big Picture
The word **"it"** in *"The animal did not cross the street because **it** was tired"*
refers to *animal*, not *street*. A Transformer learns this by letting "it" **attend** to all
other words and weight them by relevance. This notebook builds that intuition from scratch
using nothing but NumPy and random (but seeded) embeddings.

No model training happens here — this is pure intuition building.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.seed import seed_everything
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_csv
from src.utils.paths import RUNS_ATTENTION_DIR

seed_everything(42)
prepare_run_dir(RUNS_ATTENTION_DIR, clean=False)  # shared dir — don't wipe peer notebooks
print("Setup complete.")

## 2. Dataset: The Canonical Coreference Sentence

We use the famous sentence from the BERT paper that illustrates coreference resolution:

> *"The animal did not cross the street because it was tired"*

The question is: what does **"it"** refer to? The animal or the street?
A good language model should weight *animal* much higher than *street* when processing *it*.

In [ ]:
# Define the sentence and vocabulary
SENTENCE = "The animal did not cross the street because it was tired"
tokens = SENTENCE.lower().split()
n_tokens = len(tokens)

print(f"Sentence : {SENTENCE}")
print(f"Tokens   : {tokens}")
print(f"N tokens : {n_tokens}")

# Build token-to-index mapping
token2idx = {tok: i for i, tok in enumerate(tokens)}
print(f"\nToken index of 'it'     : {token2idx['it']}")
print(f"Token index of 'animal' : {token2idx['animal']}")
print(f"Token index of 'street' : {token2idx['street']}")

## 3. Architecture (Concept Diagram)

```
Query word: "it"
    │
    │  Q = embedding("it")         ← what am I looking for?
    │
    ├──── score(Q, K_animal)  ──→  high (animal is animate, can be tired)
    ├──── score(Q, K_street)  ──→  low  (streets aren't tired)
    ├──── score(Q, K_did)     ──→  low
    └──── ...                 ──→  ...
         ↓ softmax
    attention weights: [0.45, 0.02, 0.08, 0.01, 0.03, 0.05, 0.02, 0.03, 1.0, 0.25, 0.06]
         ↓ weighted sum of value vectors
    context vector for "it"
```

The **query** is what we're asking about. The **keys** are what each word
offers as a "label". The **values** are the actual content we retrieve.

## 4. Theory: The Query–Key–Value Abstraction

Think of attention as a soft database lookup:
- A **query** Q is like a search query.
- **Keys** K are the index entries in the database.
- **Values** V are the actual data stored at each entry.

The score between Q and each K determines how much of each V to retrieve:

$$\text{score}(Q, K_i) = Q \cdot K_i$$

$$\text{weight}_i = \frac{\exp(\text{score}_i)}{\sum_j \exp(\text{score}_j)}$$

$$\text{output} = \sum_i \text{weight}_i \cdot V_i$$

In practice we divide the score by $\sqrt{d_k}$ to prevent gradient vanishing (Notebook 02).

## 5. Implementation from Scratch

In [ ]:
# Create random but reproducible word embeddings
np.random.seed(42)
D_EMBED = 16  # embedding dimension

# Each token gets a random d-dimensional embedding
embeddings = np.random.randn(n_tokens, D_EMBED).astype(np.float32)

print(f"Embedding matrix shape: {embeddings.shape}  (n_tokens, d_embed)")
print(f"Embedding for 'it'    : {embeddings[token2idx['it']][:6]}...")
print(f"Embedding for 'animal': {embeddings[token2idx['animal']][:6]}...")

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def softmax(x: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    x = x - x.max()  # subtract max for numerical stability
    e = np.exp(x)
    return e / e.sum()


# Compute cosine similarity between 'it' and every other token
it_vec = embeddings[token2idx['it']]

print("Cosine similarity of 'it' with each token:")
print("-" * 45)
similarities = []
for i, tok in enumerate(tokens):
    sim = cosine_similarity(it_vec, embeddings[i])
    similarities.append(sim)
    marker = " <-- 'it' itself" if tok == 'it' else ''
    print(f"  {tok:<12} {sim:+.4f}{marker}")

In [ ]:
# Manual attention computation for 'it' attending to all tokens
# In the simplest case: Q = K = V = embedding (no learned projections yet)
query_idx = token2idx['it']  # index 8

Q = embeddings[query_idx]          # shape [D_EMBED]
K = embeddings                     # shape [n_tokens, D_EMBED]
V = embeddings                     # shape [n_tokens, D_EMBED]

# Step 1: Raw dot-product scores — Q · K^T
scores = K @ Q                     # [n_tokens]
print(f"Raw scores shape  : {scores.shape}")

# Step 2: Softmax to get attention weights
weights = softmax(scores)          # [n_tokens]
print(f"Weights shape     : {weights.shape}")
print(f"Weights sum       : {weights.sum():.6f}  (should be 1.0)")

# Step 3: Weighted sum of values → context vector
context = weights @ V              # [D_EMBED]
print(f"Context vector shape: {context.shape}")

print("\nAttention weights for 'it' attending to each token:")
print("-" * 45)
for i, (tok, w) in enumerate(zip(tokens, weights)):
    bar = '█' * int(w * 50)
    print(f"  {tok:<12} {w:.4f}  {bar}")

## 6. Sanity Checks

In [ ]:
# Sanity checks on our manual attention computation
assert embeddings.shape == (n_tokens, D_EMBED), f"Unexpected shape: {embeddings.shape}"
assert scores.shape == (n_tokens,), f"Unexpected scores shape: {scores.shape}"
assert weights.shape == (n_tokens,), f"Unexpected weights shape: {weights.shape}"
assert abs(weights.sum() - 1.0) < 1e-5, f"Weights don't sum to 1: {weights.sum()}"
assert context.shape == (D_EMBED,), f"Unexpected context shape: {context.shape}"

# 'it' should attend most to itself (diagonal is always high without learned projections)
max_idx = weights.argmax()
print(f"Token with highest attention weight: '{tokens[max_idx]}' (index {max_idx})")
print(f"Weight of 'it' attending to itself: {weights[query_idx]:.4f}")

print("\nAll sanity checks passed!")

## 7. Visualization

In [ ]:
# Plot 1: Attention heatmap for 'it' attending to all tokens
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: attention weights as a bar chart
ax = axes[0]
colors = ['tomato' if t in ['animal', 'it'] else 'steelblue' for t in tokens]
ax.bar(range(n_tokens), weights, color=colors)
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(tokens, rotation=45, ha='right')
ax.set_ylabel('Attention Weight')
ax.set_title('Attention weights: "it" attending to each token')
ax.grid(True, axis='y', alpha=0.3)

# Right: heatmap of all pairwise attention scores (every word attends to every word)
ax = axes[1]
# Compute full attention matrix: each row = one query, each col = one key
score_matrix = embeddings @ embeddings.T   # [n, n]
# Apply softmax per row
attn_matrix = np.array([softmax(row) for row in score_matrix])
im = ax.imshow(attn_matrix, cmap='Blues', aspect='auto')
ax.set_xticks(range(n_tokens))
ax.set_yticks(range(n_tokens))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(tokens, fontsize=8)
ax.set_title('Full attention matrix (row = query, col = key)')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
plot_path = RUNS_ATTENTION_DIR / 'attention_heatmap.png'
fig.savefig(plot_path, dpi=120)
plt.close(fig)
print(f"Attention heatmap saved to: {plot_path}")

In [ ]:
# Save attention scores to CSV for later analysis
rows = []
for i, tok in enumerate(tokens):
    rows.append({
        'query': 'it',
        'key': tok,
        'raw_score': float(scores[i]),
        'attention_weight': float(weights[i]),
        'cosine_similarity': float(similarities[i]),
    })

df = pd.DataFrame(rows)
csv_path = RUNS_ATTENTION_DIR / 'manual_attention_scores.csv'
df.to_csv(csv_path, index=False)
print(f"Scores saved to: {csv_path}")
print(df.to_string(index=False))

## 8. Failure Cases

**Why random embeddings don't give good coreference:**
With random embeddings there's no semantic signal. "Animal" and "it" have no special
relationship — the similarity is random noise. A trained model learns Q, K, V
projections so that semantically related words get high dot-product scores.

**What if all scores are equal?**
If Q is orthogonal to all K, scores = 0 → softmax gives uniform weights 1/N.
The context vector becomes the mean of all value vectors — no information selected.

**The scale problem:**
As D_EMBED grows, dot products grow in magnitude → softmax saturates near 0/1
→ vanishing gradients. Solution: divide by √d_k (Notebook 02).

## 9. Exercises

1. **Different query word**: Change `query_idx` to point to "animal" and re-run the attention computation. Does it attend to "tired" or "cross"?
2. **Manual softmax temperature**: Multiply scores by 0.1 before softmax. How does the attention distribution change? What about multiplying by 10?
3. **Cosine vs dot product**: Replace the dot-product scores with cosine similarities. Does the attention distribution change?
4. **Sentence variation**: Change the sentence to "The trophy did not fit in the suitcase because it was too large". Who does "it" refer to now?
5. **Value exploration**: Set V to an identity matrix (one-hot vectors). Show that the output context vector is just the weighted sum of one-hot vectors = the weighted bag-of-words.

## 10. Key Takeaways

- **Attention** lets each token in a sequence look at every other token and retrieve relevant information.
- The **query–key–value** metaphor: Q = search query, K = index, V = stored content.
- **Softmax** turns raw scores into probabilities (weights) that sum to 1.
- **Context vector** = weighted sum of value vectors — a mixture of all tokens, focused by attention.
- Random embeddings yield random attention; trained projections yield meaningful attention.
- The scaling issue (large d_k → saturated softmax) motivates the *scaled* dot-product attention in Notebook 02.